# Simulation: drogued drifter and point particle runs

Run drogued drifter and point particle simulations from observed
deployment positions. Six drifters (D298–D303) with 3 m drogues are
simulated in effective currents (Eulerian + Stokes) using
`data/cmems/effective_currents.nc`. Surface and 3 m point particles
provide baselines. All output is saved as zarr for fast downstream use.

## Parameters

In [1]:
CSV_PATH = "data/drifters_science.csv"
EFFECTIVE_CURRENTS_PATH = "data/cmems/effective_currents.nc"
OUTPUT_DIR = "output"
DROGUE_DEPTH = 3.0
DT = 300.0
RUNTIME_HOURS = 288
OUTPUTDT = 3600.0

## Imports

In [2]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from parcels import FieldSet, Particle, ParticleFile, ParticleSet, StatusCode
from parcels.kernels import AdvectionEE

from drogued_drifters.drifter import DroguedDrifter, make_dd_velocity_interpolator

/Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/w1/m9mm9h9167z_gcfzfffr0rgsh6j6kj/T/ipykernel_21713/1887068675.py:7: UserWarning: This is an alpha version of Parcels v4. The API is not stable and may change without deprecation warnings.
  from parcels import FieldSet, Particle, ParticleFile, ParticleSet, StatusCode


## Load science observations

Extract deployment position and time as the first record per drifter
within the effective currents time window (2023-04-24 onwards).

In [3]:
df = pd.read_csv(CSV_PATH, parse_dates=["date_UTC"])
drifter_ids = sorted(df["D_number"].unique())

# First record per drifter within effective-currents time window
df_window = df[df["date_UTC"] >= "2023-04-24"]
deployments = {}
for d_num in drifter_ids:
    first = df_window[df_window["D_number"] == d_num].iloc[0]
    deployments[d_num] = {
        "lon": first["Longitude"],
        "lat": first["Latitude"],
        "time": first["date_UTC"],
    }

for d_num, dep in deployments.items():
    print(f"  {d_num}: ({dep['lat']:.4f}N, {dep['lon']:.4f}E) at {dep['time']}")

  D298: (54.5093N, 10.1950E) at 2023-04-24 08:00:00
  D299: (54.5093N, 10.1961E) at 2023-04-24 08:00:00
  D300: (54.5096N, 10.1969E) at 2023-04-24 08:00:00
  D301: (54.5099N, 10.1968E) at 2023-04-24 08:00:00
  D302: (54.5096N, 10.1966E) at 2023-04-24 08:00:00
  D303: (54.5096N, 10.1968E) at 2023-04-24 08:00:00


## Load effective currents and build FieldSet

The pre-computed effective currents (`U_eff`, `V_eff`) already include
Eulerian + Stokes drift. A z=0 surface layer is prepended by copying
the shallowest level so that Parcels can interpolate from the surface.

In [4]:
ds_eff_raw = xr.open_dataset(EFFECTIVE_CURRENTS_PATH)[["U_eff", "V_eff"]].load()

# Prepend z=0 surface layer (copy of shallowest level)
ds_z0 = ds_eff_raw.isel(depth=0).assign_coords(depth=0.0)
ds_eff = xr.concat([ds_z0, ds_eff_raw], dim="depth").rename(
    {"U_eff": "U", "V_eff": "V"}
).fillna(0.0)

print(ds_eff)
print("depth levels:", ds_eff.depth.values)

<xarray.Dataset> Size: 692MB
Dimensions:    (time: 336, latitude: 150, longitude: 143, depth: 6)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2023-04-24 ... 2023-05-07T23:00:00
  * latitude   (latitude) float32 600B 53.51 53.52 53.54 ... 55.96 55.97 55.99
  * longitude  (longitude) float32 572B 9.042 9.069 9.097 ... 12.93 12.96 12.99
  * depth      (depth) float64 48B 0.0 0.5016 1.516 2.548 3.602 4.684
Data variables:
    U          (time, latitude, longitude, depth) float64 346MB 0.0 0.0 ... 0.0
    V          (time, latitude, longitude, depth) float64 346MB 0.0 0.0 ... 0.0
Attributes:
    description:  Effective current: Eulerian + Stokes drift profile
depth levels: [0.         0.50164622 1.5159924  2.54808402 3.6022985  4.68408108]


In [5]:
# Attach sgrid topology attribute required by FieldSet.from_sgrid_conventions
ds_eff["grid"] = xr.DataArray(
    data=0,
    attrs={
        "cf_role": "grid_topology",
        "topology_dimension": 2,
        "node_dimensions": "longitude latitude",
        "face_dimensions": (
            "longitude:longitude (padding: none) "
            "latitude:latitude (padding: none)"
        ),
        "vertical_dimensions": "depth:depth (padding: none)",
        "node_coordinates": "longitude latitude",
    },
)

fieldset_dd = FieldSet.from_sgrid_conventions(ds_eff, mesh="spherical")
fieldset_pp = FieldSet.from_sgrid_conventions(ds_eff, mesh="spherical")
print("FieldSets built.")

FieldSets built.


## Set up drogued drifter interpolator

Replace the default Parcels velocity interpolator with
`make_dd_velocity_interpolator`. At each RHS evaluation it extracts the
full velocity profile at each particle position and runs
`DroguedDrifter.get_final_drift_batch` to obtain the steady-state drift
velocity.

In [6]:
dd = DroguedDrifter()
fieldset_dd.UV.vector_interp_method = make_dd_velocity_interpolator(dd, spherical=True)
print("DroguedDrifter interpolator attached.")

DroguedDrifter interpolator attached.


## Helper kernel and release arrays

In [ ]:
def DeleteOOB(particles, fieldset):
    state = np.asarray(particles.state)
    oob = (state == StatusCode.ErrorOutOfBounds) | (state == StatusCode.ErrorThroughSurface)
    if np.any(oob):
        particles.state = np.where(oob, StatusCode.Delete, state)


RUNTIME = RUNTIME_HOURS * 3600
SURFACE_DEPTH = float(ds_eff.depth.isel(depth=0))  # 0.0 after prepending z=0
DROGUE_DEPTH_LEVEL = float(ds_eff.depth.sel(depth=DROGUE_DEPTH, method="nearest"))

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

release_lons = [deployments[d]["lon"] for d in drifter_ids]
release_lats = [deployments[d]["lat"] for d in drifter_ids]
release_times = [np.datetime64(deployments[d]["time"]) for d in drifter_ids]

print(f"Surface depth: {SURFACE_DEPTH} m")
print(f"Drogue depth level: {DROGUE_DEPTH_LEVEL} m")
print(f"Runtime: {RUNTIME_HOURS} h = {RUNTIME} s")

## Run drogued drifter simulation

In [8]:
dd_store = str(output_dir / "sim_drogued_drifter.zarr")
shutil.rmtree(dd_store, ignore_errors=True)

pset_dd = ParticleSet(
    fieldset=fieldset_dd,
    pclass=Particle,
    lon=release_lons,
    lat=release_lats,
    z=[SURFACE_DEPTH] * len(drifter_ids),
    time=release_times,
)
pset_dd.execute(
    kernels=[AdvectionEE, DeleteOOB],
    dt=DT,
    runtime=RUNTIME,
    output_file=ParticleFile(store=dd_store, outputdt=OUTPUTDT),
    verbose_progress=False,
)
print(f"Saved: {dd_store}")

INFO: Output files are stored in /Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/examples/baltic_drifters/output/sim_drogued_drifter.zarr


/Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/.pixi/envs/default/lib/python3.14/site-packages/parcels/_core/particlefile.py:281: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  np.less_equal(
/Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/.pixi/envs/default/lib/python3.14/site-packages/parcels/_core/particlefile.py:286: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  & np.greater_equal(
/Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/.pixi/envs/default/lib/python3.14/site-packages/parcels/_core/particlefile.py:293: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  & np.equal(time, particle_data["time"], where=np.isfinite(particle_data["time"]))


Saved: output/sim_drogued_drifter.zarr


## Run surface point particle simulation

In [9]:
surface_store = str(output_dir / "sim_surface.zarr")
shutil.rmtree(surface_store, ignore_errors=True)

pset_surface = ParticleSet(
    fieldset=fieldset_pp,
    pclass=Particle,
    lon=release_lons,
    lat=release_lats,
    z=[SURFACE_DEPTH] * len(drifter_ids),
    time=release_times,
)
pset_surface.execute(
    kernels=[AdvectionEE, DeleteOOB],
    dt=DT,
    runtime=RUNTIME,
    output_file=ParticleFile(store=surface_store, outputdt=OUTPUTDT),
    verbose_progress=False,
)
print(f"Saved: {surface_store}")

INFO: Output files are stored in /Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/examples/baltic_drifters/output/sim_surface.zarr


Saved: output/sim_surface.zarr


## Run drogue-depth point particle simulation

In [10]:
drogue_store = str(output_dir / "sim_3m.zarr")
shutil.rmtree(drogue_store, ignore_errors=True)

pset_drogue = ParticleSet(
    fieldset=fieldset_pp,
    pclass=Particle,
    lon=release_lons,
    lat=release_lats,
    z=[DROGUE_DEPTH_LEVEL] * len(drifter_ids),
    time=release_times,
)
pset_drogue.execute(
    kernels=[AdvectionEE, DeleteOOB],
    dt=DT,
    runtime=RUNTIME,
    output_file=ParticleFile(store=drogue_store, outputdt=OUTPUTDT),
    verbose_progress=False,
)
print(f"Saved: {drogue_store}")

INFO: Output files are stored in /Users/wrath/src/github.com/geomar-od-lagrange/2025_drogued_drifters/examples/baltic_drifters/output/sim_3m.zarr


Saved: output/sim_3m.zarr


## Summary

In [11]:
import os

stores = [
    ("Drogued drifter", dd_store),
    ("Surface point particle", surface_store),
    ("3 m point particle", drogue_store),
]

print("Output file sizes:")
for label, store in stores:
    total = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, dn, filenames in os.walk(store)
        for f in filenames
    )
    ds_out = xr.open_zarr(store)
    n_traj = ds_out.sizes["trajectory"]
    n_obs = ds_out.sizes["obs"]
    print(f"  {label}: {n_traj} trajectories x {n_obs} obs, {total / 1024:.1f} kB  -> {store}")

Output file sizes:
  Drogued drifter: 6 trajectories x 289 obs, 65.7 kB  -> output/sim_drogued_drifter.zarr
  Surface point particle: 6 trajectories x 289 obs, 65.7 kB  -> output/sim_surface.zarr
  3 m point particle: 6 trajectories x 289 obs, 65.7 kB  -> output/sim_3m.zarr
